In [1]:
!pip install textstat

In [2]:
!pip install calcbench-api-client

In [3]:
!pip install calcbench-api-client[BeautifulSoup,Backoff] tqdm

zsh:1: no matches found: calcbench-api-client[BeautifulSoup,Backoff]


In [4]:
import zipfile 
import pandas as pd
import re
from collections import Counter
import itertools
import time
import copy
import os

#Calc Bench
import calcbench as cb
from datetime import date, timedelta
from tqdm import tqdm, tqdm_notebook 
import requests, re
import textstat
import numpy as np
#import wrds




In [5]:
#This code is so that I know where Python is pulling data from
%pwd

'/Users/hannahnguyen/Desktop/Grant and Nguyen/Grant and Nguyen/Hannah'

In [6]:
file = zipfile.ZipFile('/Users/hannahnguyen/Desktop/VILLANOVA/Match Research/10-X_C_2016-2020.zip', 'r')

In [7]:
c = Counter(file.namelist())
filelist = file.namelist()
n = len(filelist)

In [8]:
n = len(filelist)

In [9]:
files = Counter(file.namelist())

In [10]:
n

140281

In [23]:
t0 = time.time()
filelist = [f for f in file.namelist() if "2018" in f]
n = min(10000, len(filelist))

rows = []

# General ASU-related phrases -created after reviewing Double Irish firm disclosures and a sample of approximately 15 non-DI firms 
asu_pattern = re.compile(
r"ASU\s*(No\.\s*)?2016\s*-?\s*16|"
r"Accounting\s+Standards\s+Update(\s+No\.)?\s+2016\s*-?\s*16|"
r"intra[-\s]?entity\s+(transfers?|assets?)|"
r"intra[-\s]?entity\s+asset\s+transfers?|"
r"intra[-\s]?group\s+transfers?|"
r"intercompany\s +transfers?\s+of\s+assets|"
r"intercompany\s+transfers?\s+of\s+intellectual|" 
r"intercompany\s+transfers?\s+of\s+intangible|"
r"intercompany\s+transfers?\s+of\s+certain\s+intellectual|" 
r"intercompany\s+transfers?\s+of\s+certain\s+intangible|" 
r"sold\s+intellectual\s+property\s+(between|to)|"
r"transfer\s+of\s+intangible\s+(propert(?:y|ies)|assets?)",
re.IGNORECASE
)
#NOTES: #Too broad - cathes intercompany transfer pricing : r"intercompany\s+transfer|" 
# instead - added the OF to the end so that it would cantch intercompany transfers of (not intercompany transfer pricing) 
#Below catches intra-enty transers or intra-entity assets - the latter I'm not sure is a thing (but was in someone's disclosure?) 
#r"intra[-\s]?entity\s+(transfers?|assets?)|"

#The insurance holding company statutes and regulations also require prior approval of, or in certain circumstances, prior notice to the home state regulator of, certain material intercompany transfers of assets, as well as certain transactions between insurance companies, their parent companies and affiliates.
#make sure insurance companies are dropped: LIFE INSURANCE [6311]

#DTA patterns:
#could consider adding retained earnings measures, pre-paid taxes 
dta_pattern = re.compile(
r"increase\s+(to|in)\s+deferred\s+(income\s+)?tax\s+assets?|"
r"deferred\s+(income\s+)?tax\s+assets?\s+(increased|was\s+increased|were\s+recognized|were\s+recorded)|"
r"recognized\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+asset|"
r"recorded\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+asset|"
r"generated\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+asset|"
r"resulted\s+in\s+(a|an)?\s*deferred\s+(income\s+)?tax\s+asset|"
r"to\s+record\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+asset|"
r"to\s+recognize\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+asset|"
r"step[-\s]?up\s+in\s+(tax\s+)?basis.{0,80}?deferred\s+(income\s+)?tax\s+asset|"
r"fair\s+value.*deferred\s+(income\s+)?tax\s+asset|"
r"tax\s+amortization.*deferred\s+(income\s+)?tax\s+asset|"
r"material\s+tax\s+benefit.{0,80}?deferred\s+(income\s+)?tax\s+asset",
re.IGNORECASE
)


dtl_pattern = re.compile(
r"increase\s+(to|in)\s+deferred\s+(income\s+)?tax\s+liabilit(?:y|ies)|"
r"deferred\s+(income\s+)?tax\s+liabilit(?:y|ies)\s+(increased|was\s+increased|were\s+recognized|were\s+recorded)|"
r"recognized\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+liability|"
r"recorded\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+liability|"
r"generated\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+liability|"
r"resulted\s+in\s+(a|an)?\s*deferred\s+(income\s+)?tax\s+liability|"
r"to\s+record\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+liability|"
r"to\s+recognize\s+(a|an)?\s+deferred\s+(income\s+)?tax\s+liability|"
r"step[-\s]?up\s+in\s+(tax\s+)?basis.{0,80}?deferred\s+(income\s+)?tax\s+liability|"
r"fair\s+value.*deferred\s+(income\s+)?tax\s+liability|"
r"tax\s+amortization.*deferred\s+(income\s+)?tax\s+liability|"
r"material\s+tax\s+expense.{0,80}?deferred\s+(income\s+)?tax\s+liability",
re.IGNORECASE
)

#WITHIN ASU PARAGRAH MATHCES
#\s is to precisely capture a space
#To discuss - whether to include tax benefits
#Decreaeses to DTLS? 
dta_exact = re.compile(
r"deferred[\s\-]+(income[\s\-]+)?tax(es)?[\s\-]+assets?|"
r"deferred[\s\-]+assets?|"
r"tax[\s\-]+benefits?|"
r"\bDTA\b|\bDTAs\b|"
r"deferred[\s\-]+income[\s\-]+tax(es)?",
re.IGNORECASE)

dtl_exact = re.compile(
r"deferred[\s\-]+(income[\s\-]+)?tax(es)?[\s\-]+liabilit(?:y|ies)|"
r"deferred[\s\-]+liabilit(?:y|ies)|"
r"tax[\s\-]+liabilit(?:y|ies)|"
r"\bDTL\b|\bDTLs\b|"
r"deferred[\s\-]+income[\s\-]+tax(es)?",
re.IGNORECASE)


#go back to Abott's 10k from 2018 and see why not catching this: 0000001800
# 10k satates         In October 2016, the FASB issued ASU 2016-16, Income Taxes (Topic 740): Intra-Entity Transfers of Assets Other Than Inventory, which requires the recognition of the income tax effects of intercompany sales and transfers of assets, other than inventory, in the period in which the transfer occurs. Abbott adopted the standard on January 1, 2018, using a modified retrospective approach and recorded a cumulative catch-up adjustment to Earnings employed in the business in the Consolidated Balance Sheet that was not significant.
#however, paragraph only pulling In October 2016, the FASB issued ASU 2016-16, Income Taxes (Topic 740): Intra-Entity Transfers of Assets Other Than Inventory , which requires the recognition of the income tax effects of intercompany sales and transfers of assets, other than inventory, in the period in which the transfer occurs.
no_effect_pattern = re.compile(
r"(no\s+material\s+impact)|"
r"(no\s+material\s+effect)|"
r"(did\s+not\s+(have|result\s+in)\s+(a\s+)?material\s+(impact|effect))|"
r"(not\s+expected\s+to\s+have\s+a\s+material\s+(impact|effect))|"
r"(had\s+no\s+(material\s+)?impact)|"
r"(no\s+effect)|"
r"did\s+not\s+have|"
r"not\s+significant|"
r"(do\s+(not|n't)\s+expect(\s+that)?(\s+we)?(\s+currently)?\s+(our\s+)?(pending\s+)?adoption\s+of\s+ASU\s+\d{4}[-–]?\d{2}.*?material\s+(impact|effect))",
re.IGNORECASE)
#last example of no effect is very specific to capture one of the sample firm disclosures 


#IP words used as part of ASU identification...
ip_words = re.compile(
r"\b(brand(?:s)?|copyright(?:s)?|customer\s+list(?:s)?|goodwill|"
r"intellectual\s+propert(?:y|ies)|intangible(?:s)?|IP|know[-\s]?how|"
r"license(?:s)?|patent(?:s)?|proprietary|royalt(?:y|ies)|"
r"software|technolog(?:y|ies)|trademark(?:s)?|"
r"amortization|"
r"amortize(d)?"
r"trade\s+secret(?:s)?)\b",
re.IGNORECASE)

#Should add country count! US, low-tax, non-haven
#Length of the ASU explanation - Also lots of firms talking about a "realignment"

#Beginning in 2017, some firms began talking about how they were aware of the ASU. So need to talk these firms out. 
#REMOVED GUIDANCE FROM MAYBES – LOTS SAY “ASU GUIDANCE OR IN ACCORDANCE WITH ASU GUIDANCE 

maybe_words = re.compile(
r"evaluating|"
r"anticipates|"
r"will\s+adopt|"
r"pending",
re.IGNORECASE) 

#intentionally in past tense. Still tough because of how firms describe the ASU 
#went through the file asuscrape 2016 to 2020 to identify these sentences 
recognized_words = re.compile(
r"(?:we|company)\s+recognized|"
r"(?:we|company)\s+recorded|"
r"(?:we|company)\s+implemented|"
r"(?:we|company)\s+generated|"
r"(?:we|company)\s+adopted|"
r"upon\s+adoption|"
r"adoption\s+of|"
r"were\s+adopted|"
r"also\s+adopted|"
r"(?:we|company)\s+applied|"
r"(?:we|company)\s+realized|"
r"(?:we|company)\s+completed|"
r"reorganized|"
r"resulted",
re.IGNORECASE)
    
retro = re.compile(
r"retrospective|"
r"early\s+adoption",
re.IGNORECASE) 


#good example: 2019/QTR4/20191106_10-Q_edgar_data_1145057_0001145057-19-000049.txt
#could consider tightening to say from x to y - had to do this for the US because of how often it was picking up references to US laws
Havens = r'Cyprus|Cypriot|Hong Kong|Hong Kong|Cantonese|Ireland|Irish|Lebanon|Lebanese|Luxembourg|Luxembourg|Luxembourgish|Malta|Maltese|Netherlands|Dutch|Panama|Panamanian|Puerto Rico|Puerto Rican|Singapore|Singaporean|Switzerland|Swiss|Andorra|Andorran|Anguilla|Anguillan|Antigua and Barbuda|Antiguan|Barbudan|Aruba|Aruban|The Bahamas|Bahamian|Bahamas|Bahrain|Bahraini|Barbados|Barbadian|Belize|Belizean|Bermuda|Bermudian|Bermudan|Cayman Islands|Caymanian|Costa Rica|Costa Rican|Cyprus|Cypriot|Dominica|Dominican|Gibraltar|Gibraltar|Grenada|Grenadian|Hong Kong|Hong Kong|Cantonese|Ireland|Irish|Jordan|Jordanian|Lebanon|Lebanese|Liberia|Liberian|Liechtenstein|Liechtensteiner|Luxembourg|Luxembourg|Luxembourgish|Macau|Macanese|Maldives|Maldivian|Malta|Maltese|Marshall Islands|Marshallese|Mauritius|Mauritian|Monaco|Monégasque|Monacan|Nauru|Nauruan|Netherlands|Dutch|Panama|Panamanian|Puerto Rico|Puerto Rican|Saint Kitts and Nevis|Kittitian|Nevisian|Saint Lucia|Saint Lucian|Saint Vincent and the Grenadines|Saint Vincentian|Vincentian|Samoa|Samoan|San Marino|Sammarinese|Seychelles|Seychellois|Singapore|Singaporean|Switzerland|Swiss|Tonga|Tongan|Vanuatu|Ni-Vanuatu|Vanuatuan|British Virgin Islands|British Virgin Island'
Foreign = r'Abkhazia|Abkhaz|Abkhazian|Afghanistan|Afghan|Åland Islands|Åland Island|Albania|Albanian|Algeria|Algerian|American Samoa|American Samoan|Angola|Angolan|Antarctica|Antarctic|Argentina|Argentine|Argentinian|Armenia|Armenian|Australia|Australian|Austria|Austrian|Azerbaijan|Azerbaijani|Azeri|Bangladesh|Bangladeshi|Belarus|Belarusian|Belgium|Belgian|Benin|Beninese|Beninois|Bhutan|Bhutanese|Bolivia|Bolivian|Bonaire|Bonaire|Bonairean|Bosnia and Herzegovina|Bosnian|Herzegovinian|Botswana|Botswanan|Bouvet Island|Bouvet Island|Brazil|Brazilian|Brunei|Bruneian|Bulgaria|Bulgarian|Burkina Faso|Burkinabé|Burma|Burmese|Cabo Verde|Cabo Verdean|Cambodia|Cambodian|Cameroon|Cameroonian|Canada|Canadian|Central African Republic|Central African|Chad|Chadian|Chile|Chilean|China|Chinese|Christmas Island|Christmas Island|Keeling Islands|Cocos Island|Colombia|Colombian|Comoros|Comoran|Comorian|Democratic Republic of the Congo|Congolese|Republic of the Congo|Congolese|Cook Islands|Cook Island|Croatia|Croatian|Cuba|Cuban|Curaçao|Curaçaoan|Czech Republic|Czech|Denmark|Danish|Djibouti|Djiboutian|Dominican Republic|Dominican|East Timor|Timorese|Ecuador|Ecuadorian|Egypt|Egyptian|El Salvador|Salvadoran|England|English|Equatorial Guinea|Equatorial Guinean|Equatoguinean|Eritrea|Eritrean|Estonia|Estonian|Eswatini|Swazi|Swati|Ethiopia|Ethiopian|Falkland Islands|Falkland Island|Faroe Islands|Faroese|Fiji|Fijian|Finland|Finnish|France|French|French Guiana|French Guianese|French Polynesia|French Polynesian|French Southern Territories|French Southern Territories|Gabon|Gabonese|The Gambia|Gambian|Georgia|Georgian|Germany|German|Ghana|Ghanaian|Greece|Greek|Hellenic|Greenland|Greenland|Guadeloupe|Guadeloupe|Guam|Guamanian|Guatemala|Guatemalan|Guernsey|Guernsey|Guinea|Guinean|Guinea-Bissau|Bissau-Guinean|Guyana|Guyanese|Haiti|Haitian|Heard Island and McDonald Islands|Heard Island|McDonald Island|Honduras|Honduran|Hungary|Hungarian|Magyar|Iceland|Icelandic|India|Indian|Indonesia|Indonesian|Iran|Iranian|Persian|Iraq|Iraqi|Isle of Man|Manx|Israel|Israeli|Israelite|Italy|Italian|Ivory Coast|Ivorian|Jamaica|Jamaican|Jan Mayen|Jan Mayen|Japan|Japanese|Jersey|Jersey|Kazakhstan|Kazakhstani|Kenya|Kenyan|Kiribati|Kiribati|North Korea|North Korean|South Korea|South Korean|Kosovo|Kosovar|Kosovan|Kuwait|Kuwaiti|Kyrgyzstan|Kyrgyzstani|Kyrgyz|Kirgiz|Kirghiz|Laos|Lao|Laotian|Latvia|Latvian|Lettish|Lesotho|Basotho|Libya|Libyan|Lithuania|Lithuanian|Madagascar|Malagasy|Malawi|Malawian|Malaysia|Malaysian|Mali|Malian|Malinese|Martinique|Martiniquais|Martinican|Mauritania|Mauritanian|Mayotte|Mahoran|Mexico|Mexican|Micronesia|Micronesian|Moldova|Moldovan|Mongolia|Mongolian|Montenegro|Montenegrin|Montserrat|Montserratian|Morocco|Moroccan|Mozambique|Mozambican|Myanmar|Myanma|Burmese|Namibia|Namibian|Nepal|Nepali|Nepalese|New Caledonia|New Caledonian|New Zealand|New Zealand|Nicaragua|Nicaraguan|Niger|Nigerien|Nigeria|Nigerian|Niue|Niuean|Norfolk Island|Norfolk Island|North Macedonia|Macedonian|Northern Ireland|Northern Irish|Northern Mariana Islands|Northern Marianan|Norway|Norwegian|Oman|Omani|Pakistan|Pakistani|Palau|Palauan|Palestine|Palestinian|Papua New Guinea|Papua New Guinean|Papuan|Paraguay|Paraguayan|Peru|Peruvian|Philippines|Filipino|Philippine|Pitcairn Islands|Pitcairn Island|Poland|Polish|Portugal|Portuguese|Qatar|Qatari|Réunion|Réunionese|Réunionnais|Romania|Romanian|Russia|Russian|Rwanda|Rwandan|Saba|Saban|Saint Barthélemy|Barthélemois|Saint Helena, Ascension and Tristan da Cunha|Saint Helenian|Saint Martin|Saint-Martinoise|Saint Pierre and Miquelon|Saint-Pierrais|Miquelonnais|Sahrawi|Sahrawi|Western Saharan|Sahrawian|São Tomé and Príncipe|São Toméan|Saudi Arabia|Saudi|Saudi Arabian|Scotland|Scottish|Senegal|Senegalese|Serbia|Serbian|Sierra Leone|Sierra Leonean|Sint Eustatius|Sint Eustatius|Statian|Sint Maarten|Sint Maarten|Slovakia|Slovak|Slovenia|Slovenian|Slovene|Solomon Islands|Solomon Island|Somalia|Somali|Somalian|Somaliland|Somalilander|South Africa|South African|South Georgia and the South Sandwich Islands|South Georgia Island|South Sandwich Island|South Ossetia|South Ossetian|South Sudan|South Sudanese|Spain|Spanish|Sri Lanka|Sri Lankan|Sudan|Sudanese|Suriname|Surinamese|Svalbard|Svalbard|Sweden|Swedish|Syria|Syrian|Taiwan|Taiwanese|Formosan|Tajikistan|Tajikistani|Tanzania|Tanzanian|Thailand|Thai|Timor-Leste|Timorese|Togo|Togolese|Tokelau|Tokelauan|Trinidad and Tobago|Trinidadian|Tobagonian|Tunisia|Tunisian|Turkey|Turkish|Turkmenistan|Turkmen|Turks and Caicos Islands|Turks and Caicos Island|Tuvalu|Tuvaluan|Uganda|Ugandan|Ukraine|Ukrainian|United Arab Emirates|Emirati|Emirian|Emiri|United Kingdom of Great Britain and Northern Ireland|British|United Kingdom|UK|Uruguay|Uruguayan|Uzbekistan|Uzbekistani|Uzbek|Vatican City|Vatican|Venezuela|Venezuelan|Vietnam|Vietnamese|Wales|Welsh|Wallis and Futuna|Wallis and Futuna|Wallisian|Futunan|Western Sahara|Sahrawi|Sahrawian|Sahraouian|Yemen|Yemeni|Zambia|Zambian|Zanzibar|Zanzibari|Zimbabwe|Zimbabwean'
US =  re.compile(
r"to\s+the\s+United\s+States|"
r"to\s+the\s+US|"
r"to\s+the\s+U.S.",
re.IGNORECASE) 


for i in tqdm(range(n), desc="Processing files", unit="file"):

    txtfile = file.open(filelist[i], 'r')
    text = txtfile.read().decode("utf-8", errors="ignore") 

    # Metadata
    try:
        companyname = re.search(r'COMPANY CONFORMED NAME:\t+(.+?)\r', text).group(1)
    except:
        companyname = None

    try:
        cik = re.search(r'CENTRAL INDEX KEY:\t+(.+?)\r', text).group(1)
    except:
        cik = None

    try:
        formtype = re.search(r'FORM TYPE:\t+(.+?)\r', text).group(1)
    except:
        formtype = None

    try:
        reportdate = re.search(r'CONFORMED PERIOD OF REPORT:\t(.+?)\r\n', text).group(1)
    except:
        reportdate = None

    try:
        filingdate = re.search(r'FILED AS OF DATE:\t+(.+?)\r', text).group(1)
    except:
        filingdate = None

    try:
        sich = re.search(r'STANDARD INDUSTRIAL CLASSIFICATION:\t(.+?)\r\n', text).group(1)
    except:
        sich = None

    text = text.replace('\r\n', '\n')
    reportlength = len(text)
    paragraphs = re.split(r'\n\s*\n', text)
    
    
# Defaults - IF ADDING NEW CATEGORY NEED TO UPDATE HERE 
    asu_paragraphs = []
    asu_context = ""
    asu_no_effect = False
    asu_no_effect_context = ""
    dta_paragraphs = []
    dta_context = ""
    dtl_paragraphs = []
    dtl_context = ""
    dta_asu_matches = ''
    DTA_ASU_Count = 0
    dtl_asu_matches = '' 
    DTL_ASU_Count = 0
    IP_asu_matches = '' 
    IP_ASU_Count = 0
    DTA_all_Count = 0
    DTL_all_Count = 0
    haven_asu = ""
    Haven_ASU_Count = 0 
    nonhaven_asu = ""
    Nonhaven_ASU_Count = 0
    US_asu = ""
    US_ASU_Count = 0
    recognized_ASU_matches = ""
    Recognized_ASU_Count = 0
    maybe_ASU_matches = ""
    Maybe_ASU_Count = 0
    impact_sentence = ""
    refined_asu_sentences = ""
    impact_amount = ""
    impact_date = ""


    #IF WANT TO GRAB MORE DATA (issue is that it will grab irrevlevant - so keeping with tighter match for now?)
    # Find indices where the pattern matches
    #match_indices = [i for i, p in enumerate(paragraphs) if re.search(asu_pattern, p, re.IGNORECASE)]

    # Collect the matched paragraphs and the two following ones
    #asu_paragraphs = []
    #for idx in match_indices:
    # Grab current paragraph and up to two following paragraphs, if they exist
     #   asu_paragraphs.extend(paragraphs[idx : idx + 2])

    # Remove duplicates (in case overlaps happen) and strip whitespace
    #asu_paragraphs = [p.strip() for p in dict.fromkeys(asu_paragraphs) if p.strip()]
    #asu_context = "\n\n".join(asu_paragraphs)
    match_indices = [i for i, p in enumerate(paragraphs) if asu_pattern.search(p)]
    ASUCount = len(match_indices)
    
    asu_paragraphs = []
    for idx in match_indices:
        asu_paragraphs.extend(paragraphs[idx: idx + 3])
    
    asu_paragraphs = [p.strip() for p in dict.fromkeys(asu_paragraphs) if p.strip()]
    
    target_asu = "2016-16"
    asu_number_pattern = re.compile(
        r"\b(?:ASU|Accounting Standards Update(?:\s*\(ASU\))?)(?:\s+No\.?)?\s+(\d{4}-\d{2})\b",
        re.IGNORECASE
    )
    asu_context = "\n\n".join(asu_paragraphs)
    sentences = [p.strip() for p in re.split(r'(?<=[.!?])\s+', asu_context) if p.strip()]
    
    refined_asu_sentences = []

    for sent in asu_paragraphs:
        words = sent.split()
    
        for m in asu_pattern.finditer(sent):
    
            # safer word index mapping
            prefix = sent[:m.start()]
            match_word_index = len(prefix.split())
    
            start = max(0, match_word_index - 30)
            end = min(len(words), match_word_index + 30)
    
            context = " ".join(words[start:end])
            refined_asu_sentences.append(context)

    refined_asu_sentences = list(dict.fromkeys(refined_asu_sentences))
    

    ### BG NEW CODE ABOVE - DELTED CODE FROM 3.30 version
        
    adoption_words = re.compile(
        r"\b(adopted|adoption|effective|became effective|implemented)\b",
        re.IGNORECASE
    )
    
    impact_words = re.compile(
        r"\b(resulted in|cumulative effect|recording|recorded|recognized|reclassified|adjustment|impact)\b",
        re.IGNORECASE
    )
    
    period_pattern = re.compile(
        r"\b("
        r"first quarter of \d{4}|second quarter of \d{4}|third quarter of \d{4}|fourth quarter of \d{4}|"
        r"quarter ended [A-Z][a-z]+ \d{1,2}, \d{4}|"
        r"year ended [A-Z][a-z]+ \d{1,2}, \d{4}|"
        r"fiscal year (?:beginning|ended) [A-Z][a-z]+ \d{1,2}, \d{4}|"
        r"fiscal year (?:beginning|ended) \w+ \d{1,2}, \d{4}|"
        r"annual reporting periods beginning after [A-Z][a-z]+ \d{1,2}, \d{4}|"
        r"interim reporting periods within those annual reporting periods|"
        r"as of [A-Z][a-z]+ \d{1,2}, \d{4}|"
        r"on [A-Z][a-z]+ \d{1,2}, \d{4}|"
        r"in \d{4}"
        r")\b",
        re.IGNORECASE
    )
    
    adoption_sentence = ""
    adoption_period = ""
    impact_sentence = ""
    impact_amount = ""
    impact_date = ""

    # DTA/DTL related paragraphs - in general 
    dta_paragraphs = [p.strip() for p in paragraphs if dta_pattern.search(p)]
    dta_context = "\n\n".join(dta_paragraphs)
    DTA_all_Count = len(dta_paragraphs)


    dtl_paragraphs = [p.strip() for p in paragraphs if dtl_pattern.search(p)]
    dtl_context = "\n\n".join(dtl_paragraphs)
    DTL_all_Count = len(dtl_paragraphs)

    
    if ASUCount > 0:
       #WITHIN ASU DISCLOSURE TESTS 
        # Check for "no effect" language within the ASU paragraph 
        asu_no_effect_paragraphs = [p for p in refined_asu_sentences if no_effect_pattern.search(p)]
        asu_no_effect = 1 if asu_no_effect_paragraphs else 0
        asu_no_effect_context = "\n\n".join(asu_no_effect_paragraphs)
    
        # DTA/DTL related mentions WITHIN the ASU paragrabs  
        dta_asu_matches = dta_exact.findall(asu_context)
        DTA_ASU_Count = len(dta_asu_matches)
    
        dtl_asu_matches = dtl_exact.findall(asu_context)
        DTL_ASU_Count = len(dtl_asu_matches)
    
        IP_asu_matches = ip_words.findall(asu_context)
        IP_ASU_Count = len(IP_asu_matches)
    
        haven_asu = re.findall(Havens, asu_context, re.IGNORECASE)
        Haven_ASU_Count = len(re.findall(Havens, asu_context, re.IGNORECASE))
    
        nonhaven_asu = re.findall(Foreign, asu_context, re.IGNORECASE)
        Nonhaven_ASU_Count = len(re.findall(Foreign, asu_context, re.IGNORECASE))
    
        US_asu = US.findall(asu_context)
        US_ASU_Count = len(US_asu)
    
        Recognized_ASU_Count = len(recognized_words.findall(asu_context))
    
        maybe_ASU_matches = maybe_words.findall(asu_context)
        Maybe_ASU_Count = len(maybe_ASU_matches)

    amount_pattern = re.compile(r'(\$?\s?\d+(\.\d+)?\s?(million|billion|thousand))', re.IGNORECASE)
    date_pattern = re.compile(r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}')
    
    for s in refined_asu_sentences:
        if adoption_words.search(s) and not no_effect_pattern.search(s):
            if not adoption_sentence:
                adoption_sentence = s  

            if impact_words.search(s): 
                impact_sentence = s

                amt = amount_pattern.search(s)
                if amt:
                    impact_amount = amt.group(0)

                dt = date_pattern.search(s)
                if dt:
                    impact_date = dt.group(0)

                break

    d = {
        'ASUCount': ASUCount,
        'asu_context': asu_context,
        'dta_context': dta_context,
        'dtl_context': dtl_context,
        'DTA_all_Count': DTA_all_Count,
        'DTL_all_Count': DTL_all_Count,
        'asu_no_effect': asu_no_effect,
        'Recognized_ASU_Count': Recognized_ASU_Count,
        'Maybe_ASU_Count': Maybe_ASU_Count,
        'DTA_ASU_Count': DTA_ASU_Count,
        'DTL_ASU_Count': DTL_ASU_Count,
        'IP_ASU_Count': IP_ASU_Count,
        'Haven_ASU_Count': Haven_ASU_Count,
        'Nonhaven_ASU_Count': Nonhaven_ASU_Count,
        'US_ASU_Count': US_ASU_Count,
        'reportlength': reportlength,
        'file': filelist[i],
        'companyname': companyname,
        'cik': cik,
        'formtype': formtype,
        'reportdate': reportdate,
        'filingdate': filingdate,
        'sich': sich,
        'asu_paragraphs':asu_paragraphs,
        'refined_asu_sentences':refined_asu_sentences,
        'impact_sentence': impact_sentence,
        'impact_amount': impact_amount,
        'impact_date': impact_date,
        'ASU_effect_flag': 1 if Recognized_ASU_Count > 0 and not asu_no_effect else 0,
        
    }
    rows.append(d)
    
t1 = time.time()
elapsed = t1 - t0

print(f"\n✅ Finished processing {n} files in {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


columns = [
    'ASUCount','asu_context','dta_context','dtl_context','DTA_all_Count','DTL_all_Count',
    'asu_no_effect','Recognized_ASU_Count','Maybe_ASU_Count','DTA_ASU_Count','DTL_ASU_Count',
    'IP_ASU_Count','Haven_ASU_Count','Nonhaven_ASU_Count','US_ASU_Count','reportlength',
    'file','companyname','cik','formtype','reportdate','filingdate','sich',
    'asu_paragraphs','refined_asu_sentences','impact_sentence','impact_amount','impact_date','ASU_effect_flag'
]

# Convert rows (list of dicts) to DataFrame
df = pd.DataFrame(rows, columns=columns)

Processing files: 100%|█████████████████| 10000/10000 [11:20<00:00, 14.69file/s]


✅ Finished processing 10000 files in 680.66 seconds (11.34 minutes)


In [24]:
df

,ASUCount,asu_context,dta_context,dtl_context,DTA_all_Count,DTL_all_Count,asu_no_effect,Recognized_ASU_Count,Maybe_ASU_Count,DTA_ASU_Count,...,formtype,reportdate,filingdate,sich,asu_paragraphs,refined_asu_sentences,impact_sentence,impact_amount,impact_date,ASU_effect_flag
0,0,,,,0,0,False,0,0,0,...,10-Q,20151130,20160106,RUBBER & PLASTICS FOOTWEAR [3021],[],[],,,,0
1,0,,,,0,0,False,0,0,0,...,10-K,20151231,20160330,ELECTROMEDICAL & ELECTROTHERAPEUTIC APPARATUS ...,[],[],,,,0
2,0,,,,0,0,False,0,0,0,...,10-Q,20160229,20160405,RUBBER & PLASTICS FOOTWEAR [3021],[],[],,,,0
3,0,,,,0,0,False,0,0,0,...,10-Q,20160331,20160513,ELECTROMEDICAL & ELECTROTHERAPEUTIC APPARATUS ...,[],[],,,,0
4,0,,,,0,0,False,0,0,0,...,10-K/A,20151231,20160531,"SECURITY BROKERS, DEALERS & FLOTATION COMPANIE...",[],[],,,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,0,,,,0,0,False,0,0,0,...,10-Q,20180331,20180507,BIOLOGICAL PRODUCTS (NO DIAGNOSTIC SUBSTANCES)...,[],[],,,,0
9996,0,,,,0,0,False,0,0,0,...,10-Q,20180331,20180507,PHARMACEUTICAL PREPARATIONS [2834],[],[],,,,0
9997,0,,,,0,0,False,0,0,0,...,10-Q,20180331,20180507,[6221],[],[],,,,0
9998,1,"In August 2016, the FASB issued ASU 2016-15, \...",,,0,0,1,5,1,0,...,10-Q,20180331,20180507,STATE COMMERCIAL BANKS [6022],"[In August 2016, the FASB issued ASU 2016-15, ...","[January 1, 2018 and its adoption did not have...",,,,0


In [25]:
df.to_excel("/Users/hannahnguyen/Desktop/Grant and Nguyen/Grant and Nguyen/Hannah/asu_v8.xlsx", index=False)

In [19]:
df.to_excel("asuexample.xlsx", index=False)

In [20]:
#Need to drop these for stata 
df.drop(columns=['asu_context', 'dta_context', 'dtl_context'], 
        inplace=True,
       errors='ignore')

In [ ]:
df.to_stata('asu_scrape_2016to2020v2.dta')

In [ ]:
#combining Exhibit 21 data 
def intersection(ciknum, ciknums):
    Sample = [value for value in ciknum if value in ciknums]
    return Sample